# 02 — ViT Training — ViT-Base, DeiT-Base, Swin-Base (ISIC 2019)

Training Vision Transformer models on ISIC 2019 (8-class dermatology).

**Models:**
- **ViT-Base/16** ⭐ — ancrage SOTA obligatoire (Dosovitskiy et al. ICLR 2021)
- **DeiT-Base/16** — data-efficient ViT (Touvron et al. ICML 2021)
- **Swin-Base** — ViT hiérarchique optionnel (Liu et al. ICCV 2021)

**Stratégie anti-overfitting** (Anaissi et al. 2026, arXiv:2601.00286 + ViT best practices) :

| Technique | Source | Effet |
|---|---|---|
| Focal Loss (γ=2.0) | Papier | Focalise sur les échantillons difficiles/minoritaires |
| Label Smoothing (0.1) | ViT standard | Réduit la sur-confiance |
| Mixup Augmentation (α=0.2) | ViT standard | Régularisation implicite |
| Stochastic Depth (drop_path=0.1) | ViT standard | Drop aléatoire des connexions résiduelles |
| AdamW + weight decay (0.05) | ViT standard | Meilleure régularisation que Adam |
| Linear Warmup → Cosine LR | ViT standard | Fine-tuning ViT stable |
| Strong augmentation (elastic, HSV...) | Papier | Variabilité classes minoritaires |
| WeightedRandomSampler | Papier | Corrige le déséquilibre de classes |
| Gradient Clipping (max=1.0) | Standard | Stabilité d'entraînement |
| Early Stopping (patience=10) | Standard | Arrêt avant overfitting |

**Environment:** Kaggle (GPU T4 / P100)

## 0. Kaggle Setup — Clone repo & install deps

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical

In [ ]:
!pip install -q timm albumentations loguru
!pip install -q PyDrive2

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## 0b. Download ISIC 2019 Dataset

In [ ]:
import os, requests, zipfile

DATA_DIR = '/kaggle/working/xai-vit-medical/data/isic2019'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'train_zip': 'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip',
    'train_csv': 'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv',
    'test_zip':  'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip',
    'test_csv':  'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv',
}

def download(url: str, dest: str) -> None:
    print(f'Downloading {os.path.basename(dest)} ...')
    r = requests.get(url, stream=True)
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'  done ({os.path.getsize(dest)/1e6:.1f} MB)')

def extract_and_remove(zip_path: str, extract_to: str) -> None:
    print(f'Extracting {os.path.basename(zip_path)} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    os.remove(zip_path)
    print('  done.')

train_zip = f'{DATA_DIR}/ISIC_2019_Training_Input.zip'
download(URLS['train_zip'], train_zip)
extract_and_remove(train_zip, DATA_DIR)
download(URLS['train_csv'], f'{DATA_DIR}/ISIC_2019_Training_GroundTruth.csv')

test_zip = f'{DATA_DIR}/ISIC_2019_Test_Input.zip'
download(URLS['test_zip'], test_zip)
extract_and_remove(test_zip, DATA_DIR)
download(URLS['test_csv'], f'{DATA_DIR}/ISIC_2019_Test_GroundTruth.csv')

print('Dataset ready.')

## 1. Setup & Dependencies

In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import timm

from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = '/kaggle/working/xai-vit-medical'
sys.path.insert(0, PROJECT_ROOT)

from src.data.isic_dataset import ISICDataset
from src.utils.seed import set_seed

SEED = 42
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Dependencies loaded.')

## 2. Configuration

Paramètres anti-overfitting (Anaissi et al. 2026 + ViT best practices) :

In [ ]:
# ---- Paths ----
DATA_DIR = '/kaggle/working/xai-vit-medical/data/isic2019'
SAVE_DIR = '/kaggle/working/xai-vit-medical/outputs/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Dataset ----
IMAGE_SIZE  = 224
NUM_CLASSES = 8
VAL_RATIO   = 0.25      # 75/25 split (cohérent avec CNN baseline)
NUM_WORKERS = 2

# ---- Training ----
BATCH_SIZE    = 32
EPOCHS        = 100
LR            = 1e-4    # AdamW LR pour ViT fine-tuning
WEIGHT_DECAY  = 0.05    # AdamW standard transformers
PATIENCE      = 10      # Early stopping
WARMUP_EPOCHS = 5       # Linear warmup avant cosine decay
GRAD_CLIP     = 1.0

# ---- Anti-overfitting ----
DROP_PATH_RATE  = 0.1   # Stochastic depth
LABEL_SMOOTHING = 0.1   # Label smoothing dans Focal Loss
FOCAL_GAMMA     = 2.0   # Paramètre γ de Focal Loss (papier)
MIXUP_ALPHA     = 0.2   # Coefficient Mixup

CLASS_NAMES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
CLASS_NAMES_FULL = {
    'MEL':'Melanoma',           'NV':'Naevus mélanocytaire',
    'BCC':'Carcinome basocell.','AK':'Kératose actinique',
    'BKL':'Kératose bénigne',   'DF':'Dermatofibrome',
    'VASC':'Lésion vasculaire', 'SCC':'Carcinome épidermoïde',
}

print('ViT Configuration:')
for k, v in [
    ('Image size',     f'{IMAGE_SIZE}×{IMAGE_SIZE}'),
    ('Batch size',     BATCH_SIZE),
    ('Epochs',         EPOCHS),
    ('LR (AdamW)',     LR),
    ('Weight decay',   WEIGHT_DECAY),
    ('Warmup epochs',  WARMUP_EPOCHS),
    ('Drop path',      DROP_PATH_RATE),
    ('Label smooth.',  LABEL_SMOOTHING),
    ('Focal γ',        FOCAL_GAMMA),
    ('Mixup α',        MIXUP_ALPHA),
    ('Val ratio',      VAL_RATIO),
]:
    print(f'  {k:16s}: {v}')

## 3. Dataset & DataLoaders

Augmentation **strong** (ISICDataset) :
- Transforms géométriques : flips, RandomRotate90, ShiftScaleRotate (p=0.7)
- Transforms couleur : brightness/contrast, HSV, ColorJitter (p=0.7)
- CoarseDropout (simule les artefacts dermoscopiques : poils, règlette)
- ElasticTransform / GridDistortion / OpticalDistortion (p=0.2)
- GaussNoise + blur

In [ ]:
# augmentation_strength='strong' → pipeline elastic deform complet (papier)
train_dataset = ISICDataset(
    root_dir=DATA_DIR, split='train',
    image_size=IMAGE_SIZE,
    use_albumentations=True,
    augmentation_strength='strong',
    use_official_test=True,
    val_ratio=VAL_RATIO,
)
val_dataset = ISICDataset(
    root_dir=DATA_DIR, split='val',
    image_size=IMAGE_SIZE,
    use_albumentations=True,
    use_official_test=True,
    val_ratio=VAL_RATIO,
)
test_dataset = ISICDataset(
    root_dir=DATA_DIR, split='test',
    image_size=IMAGE_SIZE,
    use_albumentations=True,
    use_official_test=True,
    val_ratio=VAL_RATIO,
)

# ---- Distribution + WeightedRandomSampler ----
class_counts = Counter(train_dataset.labels)
total = len(train_dataset)
print(f'Train : {total} images | Val : {len(val_dataset)} | Test : {len(test_dataset)}')
print('\nDistribution (train):')
for i, name in enumerate(CLASS_NAMES):
    cnt = class_counts.get(i, 0)
    bar = '█' * (cnt // 500)
    print(f'  {name:5s}: {cnt:>5d}  ({cnt/total*100:.1f}%)  {bar}')

class_weights = torch.tensor(
    [total / (NUM_CLASSES * max(class_counts[i], 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float
)
sampler = WeightedRandomSampler(
    class_weights[train_dataset.labels], num_samples=total, replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS,
    pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Class weights : {class_weights.numpy().round(2)}')

## 4. Model Architecture

In [ ]:
# ---- 4a. ViT-Base/16 ⭐ — ancrage SOTA obligatoire ----
# pretrained sur ImageNet-21k → ImageNet-1k
# drop_path_rate=0.1 : stochastic depth (régularisation ViT)

def build_vit_base(
    num_classes: int = NUM_CLASSES,
    pretrained: bool = True,
    drop_path_rate: float = DROP_PATH_RATE,
) -> nn.Module:
    model = timm.create_model(
        'vit_base_patch16_224',
        pretrained=pretrained,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'ViT-Base/16  (pretrained={pretrained}, drop_path={drop_path_rate})')
    print(f'  Classifier       : {model.head}')
    print(f'  Params total     : {total:,}')
    print(f'  Params trainable : {trainable:,}')
    return model

_m = build_vit_base()
_x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _m(_x).shape == (2, NUM_CLASSES)
del _m, _x; torch.cuda.empty_cache()
print('Architecture OK.')

In [ ]:
# ---- 4b. DeiT-Base/16 — ViT data-efficient ----
# Préentraîné avec distillation → meilleure généralisation sur données limitées

def build_deit_base(
    num_classes: int = NUM_CLASSES,
    pretrained: bool = True,
    drop_path_rate: float = DROP_PATH_RATE,
) -> nn.Module:
    model = timm.create_model(
        'deit_base_patch16_224',
        pretrained=pretrained,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'DeiT-Base/16 (pretrained={pretrained}, drop_path={drop_path_rate})')
    print(f'  Classifier       : {model.head}')
    print(f'  Params total     : {total:,}')
    print(f'  Params trainable : {trainable:,}')
    return model

_m = build_deit_base()
_x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _m(_x).shape == (2, NUM_CLASSES)
del _m, _x; torch.cuda.empty_cache()
print('Architecture OK.')

In [ ]:
# ---- 4c. Swin-Base — ViT hiérarchique (optionnel) ----
# Fenêtres d'attention locales → plus efficace sur grandes images
# Supporte aussi drop_path_rate

def build_swin_base(
    num_classes: int = NUM_CLASSES,
    pretrained: bool = True,
    drop_path_rate: float = DROP_PATH_RATE,
) -> nn.Module:
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=pretrained,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Swin-Base    (pretrained={pretrained}, drop_path={drop_path_rate})')
    print(f'  Classifier       : {model.head.fc}')
    print(f'  Params total     : {total:,}')
    print(f'  Params trainable : {trainable:,}')
    return model

_m = build_swin_base()
_x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _m(_x).shape == (2, NUM_CLASSES)
del _m, _x; torch.cuda.empty_cache()
print('Architecture OK.')

## 5. Training Loop

| Composant | Choix | Raison |
|-----------|-------|--------|
| Loss | Focal Loss + Label Smoothing | Déséquilibre classes + sur-confiance |
| Optimizer | AdamW (no decay bias/norm) | Standard transformers |
| Scheduler | Linear warmup → Cosine | Fine-tuning ViT stable |
| Régularisation | drop_path + mixup + strong aug | Couches anti-overfitting multiples |
| Mixed Precision | torch.amp FP16 | Vitesse + mémoire |
| Gradient Clip | max_norm=1.0 | Stabilité |
| Early Stopping | patience=10 sur val_loss | |


In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss + Label Smoothing (Anaissi et al. 2026).
    L = -(1-pt)^γ * log(pt) avec soft targets.
    """
    def __init__(
        self,
        gamma: float = FOCAL_GAMMA,
        label_smoothing: float = LABEL_SMOOTHING,
        num_classes: int = NUM_CLASSES,
    ) -> None:
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.num_classes     = num_classes

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            smooth = torch.full_like(logits, self.label_smoothing / (self.num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        log_probs = F.log_softmax(logits, dim=1)
        probs     = torch.exp(log_probs)
        pt        = (probs * smooth).sum(dim=1)
        return ((1.0 - pt) ** self.gamma * -(smooth * log_probs).sum(dim=1)).mean()


def mixup_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = MIXUP_ALPHA,
    num_classes: int = NUM_CLASSES,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Mixup intra-batch. Retourne (images_mixées, labels_soft, labels_originaux)."""
    if alpha <= 0:
        return images, F.one_hot(labels, num_classes).float(), labels
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    lbl_a = F.one_hot(labels,      num_classes).float()
    lbl_b = F.one_hot(labels[idx], num_classes).float()
    return mixed, lam * lbl_a + (1 - lam) * lbl_b, labels


print(f'FocalLoss  : γ={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING}')
print(f'Mixup      : α={MIXUP_ALPHA}')

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    use_mixup: bool = True,
) -> tuple[float, float]:
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels, _ in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_mixup:
            images, mixed_labels, orig_labels = mixup_batch(images, labels)
            mixed_labels = mixed_labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            if use_mixup:
                loss = -(mixed_labels * F.log_softmax(outputs, dim=1)).sum(dim=1).mean()
            else:
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds      = outputs.max(1)
        ref           = orig_labels if use_mixup else labels
        correct      += preds.eq(ref).sum().item()
        total        += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels, _ in tqdm(loader, desc='  Eval ', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds      = outputs.max(1)
        correct      += preds.eq(labels).sum().item()
        total        += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    patience: int = PATIENCE,
    device: torch.device = DEVICE,
    use_mixup: bool = True,
) -> tuple[nn.Module, dict]:
    """Pipeline complet : AdamW + warmup/cosine + Focal Loss + Mixup + AMP + early stopping."""
    model = model.to(device)
    criterion = FocalLoss()

    # AdamW : pas de weight_decay sur bias, norm, cls_token, pos_embed
    no_decay = {'bias', 'norm', 'cls_token', 'pos_embed', 'patch_embed'}
    param_groups = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and p.requires_grad],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and p.requires_grad],
         'weight_decay': 0.0},
    ]
    optimizer = optim.AdamW(param_groups, lr=lr)

    warmup    = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                         total_iters=WARMUP_EPOCHS)
    cosine    = CosineAnnealingLR(optimizer,
                                   T_max=max(epochs - WARMUP_EPOCHS, 1),
                                   eta_min=lr * 0.01)
    scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])
    scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    best_val_loss = float('inf')
    no_improve    = 0
    history       = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    ckpt_path     = os.path.join(SAVE_DIR, f'{model_name}_best.pth')

    print(f'\n{"="*68}')
    print(f'  {model_name}  |  {epochs} epochs  |  device={device}')
    print(f'  mixup={use_mixup}  drop_path={DROP_PATH_RATE}  '
          f'focal_γ={FOCAL_GAMMA}  smooth={LABEL_SMOOTHING}')
    print(f'{"="*68}')

    for epoch in range(1, epochs + 1):
        t_loss, t_acc       = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, use_mixup)
        v_loss, v_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        lr_now = optimizer.param_groups[0]['lr']
        tag = ''
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            no_improve    = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': v_loss,
                'val_acc':  v_acc,
                'history':  history,
                'class_names': CLASS_NAMES,
                'model_name': model_name,
            }, ckpt_path)
            tag = ' ★'
        else:
            no_improve += 1

        print(f'  Epoch {epoch:3d}/{epochs} | '
              f'train={t_loss:.4f}/{t_acc:.4f} | '
              f'val={v_loss:.4f}/{v_acc:.4f} | '
              f'lr={lr_now:.2e}{tag}')

        if no_improve >= patience:
            print(f'\n  Early stopping at epoch {epoch} (no improvement for {patience} epochs).')
            break

    # Restaurer le meilleur checkpoint
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'\n  Best val_loss={best_val_loss:.4f} — checkpoint: {ckpt_path}')
    return model, history


print('Training functions defined.')

## 6. Train ViT-Base/16 ⭐

In [ ]:
vit_model = build_vit_base(num_classes=NUM_CLASSES, pretrained=True,
                            drop_path_rate=DROP_PATH_RATE)

vit_model, vit_history = train_model(
    model=vit_model,
    model_name='vit_base_patch16',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, device=DEVICE, use_mixup=True,
)

In [ ]:
f = drive.CreateFile({'title': 'vit_base_patch16_best.pth'})
f.SetContentFile(os.path.join(SAVE_DIR, 'vit_base_patch16_best.pth'))
f.Upload()
print(f'Uploaded vit_base_patch16_best.pth  (id={f["id"]})')

## 7. Train DeiT-Base/16

In [ ]:
deit_model = build_deit_base(num_classes=NUM_CLASSES, pretrained=True,
                              drop_path_rate=DROP_PATH_RATE)

deit_model, deit_history = train_model(
    model=deit_model,
    model_name='deit_base_patch16',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, device=DEVICE, use_mixup=True,
)

In [ ]:
f = drive.CreateFile({'title': 'deit_base_patch16_best.pth'})
f.SetContentFile(os.path.join(SAVE_DIR, 'deit_base_patch16_best.pth'))
f.Upload()
print(f'Uploaded deit_base_patch16_best.pth  (id={f["id"]})')

## 8. Train Swin-Base (optionnel)

In [ ]:
# Swin-Base : ViT hiérarchique — optionnel selon budget GPU
# Commenter cette cellule si pas assez de temps/mémoire

swin_model = build_swin_base(num_classes=NUM_CLASSES, pretrained=True,
                              drop_path_rate=DROP_PATH_RATE)

swin_model, swin_history = train_model(
    model=swin_model,
    model_name='swin_base',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, device=DEVICE, use_mixup=True,
)

In [ ]:
f = drive.CreateFile({'title': 'swin_base_best.pth'})
f.SetContentFile(os.path.join(SAVE_DIR, 'swin_base_best.pth'))
f.Upload()
print(f'Uploaded swin_base_best.pth  (id={f["id"]})')

## 9. Evaluation & Results

In [ ]:
def plot_training_curves(history: dict, model_name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(history['train_loss']) + 1)

    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val')
    axes[0].set(xlabel='Epoch', ylabel='Loss', title=f'{model_name} — Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, history['train_acc'], label='Train')
    axes[1].plot(ep, history['val_acc'],   label='Val')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy', title=f'{model_name} — Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(SAVE_DIR, f'{model_name}_curves.png')
    plt.savefig(out, dpi=150); plt.show()

    # Diagnostic overfitting
    gap = max(history['train_acc']) - max(history['val_acc'])
    status = 'overfitting detecte' if gap > 0.10 else 'gap acceptable'
    print(f'  best_train={max(history["train_acc"]):.4f}  '
          f'best_val={max(history["val_acc"]):.4f}  '
          f'gap={gap:.4f} ({status})')


def evaluate_full(
    model: nn.Module,
    loader: DataLoader,
    model_name: str,
    split: str = 'val',
    device: torch.device = DEVICE,
) -> dict:
    loss, acc, preds, labels = evaluate(model, loader, FocalLoss(), device)
    print(f'\n{model_name} — {split.upper()}')
    print(f'  Loss     : {loss:.4f}')
    print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set(xlabel='Predicted', ylabel='True',
           title=f'{model_name} — Confusion Matrix ({split})')
    plt.tight_layout()
    out = os.path.join(SAVE_DIR, f'{model_name}_cm_{split}.png')
    plt.savefig(out, dpi=150); plt.show()
    return {'loss': loss, 'acc': acc, 'preds': preds, 'labels': labels}

In [ ]:
# ---- ViT-Base ----
print('=== ViT-Base/16 ===')
plot_training_curves(vit_history, 'ViT-Base/16')
vit_val_results  = evaluate_full(vit_model,  val_loader,  'ViT-Base_16', 'val')
vit_test_results = evaluate_full(vit_model,  test_loader, 'ViT-Base_16', 'test')

In [ ]:
# ---- DeiT-Base ----
print('=== DeiT-Base/16 ===')
plot_training_curves(deit_history, 'DeiT-Base/16')
deit_val_results  = evaluate_full(deit_model, val_loader,  'DeiT-Base_16', 'val')
deit_test_results = evaluate_full(deit_model, test_loader, 'DeiT-Base_16', 'test')

In [ ]:
# ---- Swin-Base (si entraîné) ----
# Commenter si Swin-Base n'a pas été entraîné
print('=== Swin-Base ===')
plot_training_curves(swin_history, 'Swin-Base')
swin_val_results  = evaluate_full(swin_model, val_loader,  'Swin-Base', 'val')
swin_test_results = evaluate_full(swin_model, test_loader, 'Swin-Base', 'test')

## 10. Save Summary & Upload to Google Drive

In [ ]:
def load_epoch(model_name: str) -> int:
    ckpt = torch.load(os.path.join(SAVE_DIR, f'{model_name}_best.pth'), map_location='cpu')
    return int(ckpt.get('epoch', -1))

summary = {
    'vit_base_patch16': {
        'val_loss':  float(vit_val_results['loss']),
        'val_acc':   float(vit_val_results['acc']),
        'test_loss': float(vit_test_results['loss']),
        'test_acc':  float(vit_test_results['acc']),
        'best_epoch': load_epoch('vit_base_patch16'),
        'epochs_trained': len(vit_history['train_loss']),
    },
    'deit_base_patch16': {
        'val_loss':  float(deit_val_results['loss']),
        'val_acc':   float(deit_val_results['acc']),
        'test_loss': float(deit_test_results['loss']),
        'test_acc':  float(deit_test_results['acc']),
        'best_epoch': load_epoch('deit_base_patch16'),
        'epochs_trained': len(deit_history['train_loss']),
    },
    # Décommenter si Swin entraîné :
    # 'swin_base': {
    #     'val_loss':  float(swin_val_results['loss']),
    #     'val_acc':   float(swin_val_results['acc']),
    #     'test_loss': float(swin_test_results['loss']),
    #     'test_acc':  float(swin_test_results['acc']),
    #     'best_epoch': load_epoch('swin_base'),
    #     'epochs_trained': len(swin_history['train_loss']),
    # },
}

summary_path = os.path.join(SAVE_DIR, 'vit_training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('ViT Training Summary:')
print(f'{"Model":<22} {"val_acc":>8} {"test_acc":>9} {"best_ep":>8}')
print('-' * 50)
for name, m in summary.items():
    print(f'{name:<22} {m["val_acc"]:>8.4f} {m["test_acc"]:>9.4f} {m["best_epoch"]:>8}')

In [ ]:
# Upload résultats vers Google Drive
files_to_upload = [
    'vit_training_summary.json',
    'ViT-Base_16_curves.png', 'ViT-Base_16_cm_val.png', 'ViT-Base_16_cm_test.png',
    'DeiT-Base_16_curves.png', 'DeiT-Base_16_cm_val.png', 'DeiT-Base_16_cm_test.png',
    # 'Swin-Base_curves.png', 'Swin-Base_cm_val.png', 'Swin-Base_cm_test.png',
]

for fname in files_to_upload:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        f = drive.CreateFile({'title': fname})
        f.SetContentFile(fpath)
        f.Upload()
        print(f'Uploaded: {fname}  (id={f["id"]})')
    else:
        print(f'Skipped (not found): {fname}')

print('\nDone. Checkpoints ready for 03_train_dino.ipynb')